# Semana 9 — SQL para Data Warehouse

Nesta semana, vamos sair da etapa de construção do modelo dimensional e usar o Star Schema para responder perguntas de BI com SQL.

## Objetivos da Semana 9

Ao final das 3 aulas, você deverá conseguir:

- Usar `SELECT`, `WHERE` e `ORDER BY` em consultas analíticas.
- Fazer `INNER JOIN` entre tabela fato e dimensões.
- Usar `LEFT JOIN` para investigar ausência de relacionamento.
- Criar métricas com `GROUP BY`, `SUM`, `COUNT` e `AVG`.
- Filtrar agregações com `HAVING`.
- Usar funções de data como `DATE_TRUNC` e `EXTRACT`.

## Aula 1 da Semana 9 — SQL básico em DW

### Roteiro teórico

Em um Data Warehouse, o SQL é usado principalmente para análise.  
Diferente de um sistema transacional, onde o foco é inserir e atualizar registros, aqui o foco é consultar, agregar e transformar dados em indicadores.

A estrutura básica de uma consulta é:

```sql
SELECT colunas
FROM tabela
WHERE condição
ORDER BY coluna;
```

No nosso modelo:

- `fato_vendas` guarda os eventos mensuráveis de venda.
- `dim_cliente` explica quem comprou e onde.
- `dim_produto` explica o que foi vendido.
- `dim_tempo` permite analisar a venda no tempo.

### Ler dados dimensão e fato

In [9]:
%pip install pandas
%pip install pandas
%pip install duckdb

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [10]:
### Importar pacotes

import pandas as pd
import duckdb

In [11]:
dim_cliente = pd.read_csv(r'C:/Users/robso/Documents/GitHub/SCTEC/Semana_9/Dataset_schema/dim_cliente.csv')
dim_produto = pd.read_csv(r'C:/Users/robso/Documents/GitHub/SCTEC/Semana_9/Dataset_schema/dim_produto.csv')
dim_tempo = pd.read_csv(r'C:/Users/robso/Documents/GitHub/SCTEC/Semana_9/Dataset_schema/dim_tempo.csv')
fato_vendas = pd.read_csv(r'C:/Users/robso/Documents/GitHub/SCTEC/Semana_9/Dataset_schema/fato_vendas.csv')    

In [12]:
banco_dados = duckdb.connect()
banco_dados.register('dim_cliente', dim_cliente) # registra a dimensão cliente no banco de dados
banco_dados.register('dim_produto', dim_produto) # registra a dimensão produto no banco de dados
banco_dados.register('dim_tempo', dim_tempo) # registra a dimensão tempo no banco de dados
banco_dados.register('fato_vendas', fato_vendas) # registra a tabela fato vendas no banco de dados

## Consulta exemplo — Faturamento total

In [13]:
banco_dados.sql("""
SELECT 
SUM(valor_total) AS faturamento_total
FROM fato_vendas
""").df() # consulta para calcular o faturamento total da loja, somando a coluna valor_total da tabela fato_vendas

,faturamento_total
0,1.541977e+07


### Exemplo — selecionando vendas da fato

In [14]:
banco_dados.sql("""
SELECT
    venda_sk,
    order_id,
    valor_produto,
    valor_frete,
    valor_total
FROM fato_vendas
LIMIT 20
""").df()

,venda_sk,order_id,valor_produto,valor_frete,valor_total
0,1,00010242fe8c5a6d1ba2dd792cb16214,58.90,13.29,72.19
1,2,00018f77f2f0320c557190d7a144bdd3,239.90,19.93,259.83
2,3,000229ec398224ef6ca0657da4fc703e,199.00,17.87,216.87
3,4,00024acbcdf0a6daa1e931b038114c75,12.99,12.79,25.78
4,5,00042b26cf59d7ce69dfabb4e55b4fd9,199.90,18.14,218.04
5,6,00048cc3ae777c65dbb7d2a0634bc1ea,21.90,12.69,34.59
6,7,00054e8431b9d7675808bcb819fb4a32,19.90,11.85,31.75
7,8,000576fe39319847cbb9d288c5617fa6,810.00,70.75,880.75
8,9,0005a1a1728c9d785b8e2b08b904576c,145.95,11.65,157.60
9,10,0005f50442cb953dcd1d21e1fb923495,53.99,11.40,65.39


### Exemplo — usando `WHERE` e `ORDER BY`

A consulta abaixo busca vendas com valor total maior que 200 e ordena da maior para a menor.

In [15]:
banco_dados.sql("""
SELECT
    venda_sk,
    order_id,
    valor_produto,
    valor_frete,
    valor_total
FROM fato_vendas
WHERE valor_total > 200
ORDER BY valor_total ASC
""").df()

,venda_sk,order_id,valor_produto,valor_frete,valor_total
0,106711,f7f3e95cf456d7a1ebede3e7c3cce324,161.42,38.60,200.02
1,24976,3a1f0844864136aa18f6710238f40e6e,161.42,38.60,200.02
2,18378,2b31af271f3efcfd1f240a6ba82c631e,161.42,38.60,200.02
3,69955,a31cd1f2ccad0246f0517f09035a1d36,185.00,15.02,200.02
4,22353,3431f838d49b42723b060fa367b3d294,189.90,10.14,200.04
...,...,...,...,...,...
17622,10998,199af31afc78c699f0dbf71fb178d4d4,4690.00,74.34,4764.34
17623,72721,a96610ab360d42a2e5335a3998b4718a,4799.00,151.34,4950.34
17624,105496,f5136e38d1a14a4dbd87dff67da82701,6499.00,227.66,6726.66
17625,109785,fefacc66af859508bf1a7934eab1e97f,6729.00,193.21,6922.21


## Exercício 13 🟢 Fácil — SELECT em tabela fato

Liste as colunas `venda_sk`, `order_id`, `valor_produto`, `valor_frete` e `valor_total` das 15 primeiras vendas.

In [16]:
banco_dados.sql("""
SELECT
    venda_sk,
    order_id,
    valor_produto,
    valor_frete,
    valor_total
FROM
    fato_vendas
LIMIT 15        
""").df()

,venda_sk,order_id,valor_produto,valor_frete,valor_total
0,1,00010242fe8c5a6d1ba2dd792cb16214,58.90,13.29,72.19
1,2,00018f77f2f0320c557190d7a144bdd3,239.90,19.93,259.83
2,3,000229ec398224ef6ca0657da4fc703e,199.00,17.87,216.87
3,4,00024acbcdf0a6daa1e931b038114c75,12.99,12.79,25.78
4,5,00042b26cf59d7ce69dfabb4e55b4fd9,199.90,18.14,218.04
5,6,00048cc3ae777c65dbb7d2a0634bc1ea,21.90,12.69,34.59
6,7,00054e8431b9d7675808bcb819fb4a32,19.90,11.85,31.75
7,8,000576fe39319847cbb9d288c5617fa6,810.00,70.75,880.75
8,9,0005a1a1728c9d785b8e2b08b904576c,145.95,11.65,157.60
9,10,0005f50442cb953dcd1d21e1fb923495,53.99,11.40,65.39


## Exercício 14 🟢 Fácil — Filtro com WHERE

Liste as vendas cujo `valor_frete` seja maior que 50.  
Mostre `venda_sk`, `order_id`, `valor_frete` e `valor_total`.

In [17]:
banco_dados.sql("""
SELECT
    venda_sk,
    order_id,
    valor_frete,
    valor_total
FROM
    fato_vendas
WHERE
    valor_frete > 50
ORDER BY valor_frete DESC
LIMIT 20
""").df()

,venda_sk,order_id,valor_frete,valor_total
0,71885,a77e1550db865202c56b19ddc6dc4d53,409.68,1388.68
1,27413,3fde74c28a3d5d618c00f26d51baafa0,375.28,2713.36
2,3232,076d1555fb53a89b0ef4d529e527a0f6,375.28,2713.36
3,68275,9f49bd16053df810384e793386312674,339.59,1488.59
4,16360,264a7e199467906c0727394df82d1a6a,338.30,1388.30
5,86056,c7a07ddd52bbe18b61da49a8d89853d3,322.10,1372.10
6,4939,0b6230647ed16f4b3e70282dc4b5b87f,321.88,1371.88
7,3510,0822bcde10bb5d023755a71bc8f7797f,321.46,1311.46
8,29114,43bdbd9dc0931d72befdf4765af6c442,317.47,3406.47
9,47246,6ddfbf514959b49b6410c01ad93054bb,314.40,1359.40


## Exercício 15 🟡 Médio — Ordenação para ranking

Liste as 20 vendas de maior `valor_total`.  
Mostre `venda_sk`, `order_id`, `valor_produto`, `valor_frete` e `valor_total`.

In [18]:
banco_dados.sql("""
SELECT
    venda_sk,
    order_id,
    valor_produto,
    valor_frete,
    valor_total
FROM
    fato_vendas
ORDER BY 
    valor_total DESC
LIMIT 20
""").df()

,venda_sk,order_id,valor_produto,valor_frete,valor_total
0,3482,0812eb902a67711a1cb742b3cdaa65ae,6735.00,194.31,6929.31
1,109785,fefacc66af859508bf1a7934eab1e97f,6729.00,193.21,6922.21
2,105496,f5136e38d1a14a4dbd87dff67da82701,6499.00,227.66,6726.66
3,72721,a96610ab360d42a2e5335a3998b4718a,4799.00,151.34,4950.34
4,10998,199af31afc78c699f0dbf71fb178d4d4,4690.00,74.34,4764.34
5,60712,8dbc85d1447242f3b127dda390d56e19,4590.00,91.78,4681.78
6,28537,426a9742b533fc6fed17d1fd6d143d7e,4399.87,113.45,4513.32
7,55421,80dfedb6d17bf23539beeef3c768f4d7,3999.00,195.76,4194.76
8,44822,68101694e5c5dc7330c91e1bbc36214f,4099.99,75.27,4175.26
9,76618,b239ca7cd485940b31882363b52e6674,4059.00,104.51,4163.51


##  JOINS em Star Schema

### Roteiro teórico

Em um Star Schema, normalmente a análise começa na tabela fato e depois se conecta às dimensões.

Exemplo de raciocínio:

> Quero faturamento por estado.  
> O faturamento está na `fato_vendas`.  
> O estado está na `dim_cliente`.  
> Então preciso fazer JOIN entre fato e dimensão cliente.

Tipos importantes:

- `INNER JOIN`: retorna apenas registros que possuem correspondência nos dois lados.
- `LEFT JOIN`: mantém todos os registros da tabela da esquerda, mesmo sem correspondência na direita.

### Exemplo — fato + dimensão cliente

In [19]:
banco_dados.sql("""
SELECT
    f.venda_sk,
    f.order_id,
    f.valor_total,
    c.customer_state
FROM fato_vendas f
INNER JOIN dim_cliente c
    ON f.cliente_sk = c.cliente_sk
LIMIT 10
""").df()

,venda_sk,order_id,valor_total,customer_state
0,1,00010242fe8c5a6d1ba2dd792cb16214,72.19,RJ
1,2,00018f77f2f0320c557190d7a144bdd3,259.83,SP
2,3,000229ec398224ef6ca0657da4fc703e,216.87,MG
3,4,00024acbcdf0a6daa1e931b038114c75,25.78,SP
4,5,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,SP
5,6,00048cc3ae777c65dbb7d2a0634bc1ea,34.59,MG
6,7,00054e8431b9d7675808bcb819fb4a32,31.75,SP
7,8,000576fe39319847cbb9d288c5617fa6,880.75,SP
8,9,0005a1a1728c9d785b8e2b08b904576c,157.60,SP
9,10,0005f50442cb953dcd1d21e1fb923495,65.39,SP


### Exemplo — fato + dimensão produto + dimensão tempo

In [20]:
banco_dados.sql("""
SELECT
    f.venda_sk,
    f.order_id,
    p.product_category_name_english AS categoria,
    t.ano_mes,
    f.valor_total,
    c.customer_state AS estado
FROM fato_vendas f
INNER JOIN dim_produto p
    ON f.produto_sk = p.produto_sk
INNER JOIN dim_tempo t
    ON f.tempo_sk = t.tempo_sk
INNER JOIN dim_cliente c
    ON f.cliente_sk = c.cliente_sk
LIMIT 20
""").df()

,venda_sk,order_id,categoria,ano_mes,valor_total,estado
0,102401,edc438144eedf58c321995efd0225e19,construction_tools_construction,2018-07,68.45,GO
1,102402,edc531c3153d23614049077e7e1e6405,perfumery,2018-04,136.40,MG
2,102403,edc577111e3237868aec11a4da6611d9,health_beauty,2018-07,61.58,MG
3,102404,edc5c94770796f4212d272830dd17840,cool_stuff,2018-01,42.58,SP
4,102405,edc6d3b93ee7050db4013694b93176ae,sports_leisure,2017-05,30.86,SP
5,102406,edc7444663af179dd113423c0512b4e4,bed_bath_table,2018-02,67.98,SP
6,102407,edc75396403e226e7ac4e42085860602,baby,2017-05,185.93,CE
7,102408,edc852e0d271566f7923b10c089c30ec,stationery,2018-08,133.42,RS
8,102409,edc89efa4164df7e21f701a9fb88bc98,fashion_bags_accessories,2017-11,55.15,SP
9,102410,edc89efa4164df7e21f701a9fb88bc98,fashion_bags_accessories,2017-11,73.00,SP


In [21]:
banco_dados.sql("""
-- esta análise tem como objetivo identificar os 10 pedidos com maior valor total, juntamente com as informações do cliente e do estado de entrega. A consulta utiliza uma junção entre a tabela fato_vendas e a tabela dim_cliente para obter os detalhes do cliente, e ordena os resultados pelo valor total em ordem decrescente, limitando a exibição aos 10 primeiros registros.
SELECT
    v.venda_sk AS codigo_venda,
    v.order_id,
    v.cliente_sk,
    c.customer_city AS Cidade,
    c.customer_state AS Estado,
    v.valor_total
FROM fato_vendas v
    LEFT JOIN dim_cliente c
ON v.cliente_sk = c.cliente_sk
    ORDER BY v.valor_total ASC
 LIMIT 10;
 """).df()

,codigo_venda,order_id,cliente_sk,Cidade,Estado,valor_total
0,61577,8fbcb92faf1aa60361f61ed7ae721a7e,40877,teofilo otoni,MG,6.08
1,61573,8fbcb92faf1aa60361f61ed7ae721a7e,40877,teofilo otoni,MG,6.08
2,61574,8fbcb92faf1aa60361f61ed7ae721a7e,40877,teofilo otoni,MG,6.08
3,61575,8fbcb92faf1aa60361f61ed7ae721a7e,40877,teofilo otoni,MG,6.08
4,48999,71e22e2d99081d6dc07d9627bb85969e,69939,sao paulo,SP,7.28
5,56044,8272b63d03f5f79c56e9e4120aec44ef,47121,sao paulo,SP,9.09
6,56039,8272b63d03f5f79c56e9e4120aec44ef,47121,sao paulo,SP,9.09
7,56042,8272b63d03f5f79c56e9e4120aec44ef,47121,sao paulo,SP,9.09
8,56041,8272b63d03f5f79c56e9e4120aec44ef,47121,sao paulo,SP,9.09
9,56043,8272b63d03f5f79c56e9e4120aec44ef,47121,sao paulo,SP,9.09


# AULA 2 - SEMANA 9

Agregação de tabelas:
- GROUP_BY, COUNT, SUM e AVG
- WHERE vs HAVING

#### Exemplo 1 (slide 18) - Faturamento por estado

In [22]:
banco_dados.sql("""
    SELECT
        c.customer_state AS estado,
        SUM(f.valor_total) AS faturamento_total,
        COUNT(f.order_id) AS qtd_pedidos,
        AVG(f.valor_total) AS valor_medio_item
    FROM fato_vendas f
    INNER JOIN dim_cliente c
        ON f.cliente_sk = c.cliente_sk
    GROUP BY c.customer_state
    ORDER BY faturamento_total DESC;
""").df()
   

,estado,faturamento_total,qtd_pedidos,valor_medio_item
0,SP,5769703.15,46448,124.218549
1,RJ,2055401.57,14143,145.329956
2,MG,1818891.67,12916,140.824688
3,RS,861472.79,6134,140.442255
4,PR,781708.80,5649,138.380032
5,SC,595127.78,4097,145.259404
6,BA,591137.81,3683,160.504428
7,DF,346123.35,2355,146.973822
8,GO,334212.35,2277,146.777492
9,ES,317657.93,2225,142.767609


#### Exemplo 2 (slide 19) - Faturamento por categoria

In [23]:
banco_dados.sql("""
    SELECT
        p.product_category_name_english AS categoria,
        SUM(f.valor_total) AS faturamento_total,
        COUNT(*) AS qtd_itens,
        AVG(f.valor_total) AS ticket_medio
    FROM fato_vendas f
    INNER JOIN dim_produto p
        ON f.produto_sk = p.produto_sk
    GROUP BY p.product_category_name_english
    ORDER BY faturamento_total DESC;
 """).df()

,categoria,faturamento_total,qtd_itens,ticket_medio
0,health_beauty,1412089.53,9465,149.190653
1,watches_gifts,1264333.12,5859,215.793330
2,bed_bath_table,1225209.26,10953,111.860610
3,sports_leisure,1118256.91,8431,132.636331
4,computers_accessories,1032723.77,7644,135.102534
...,...,...,...,...
67,flowers,1598.91,33,48.451818
68,home_comfort_2,1170.58,30,39.019333
69,cds_dvds_musicals,954.99,14,68.213571
70,fashion_childrens_clothes,598.67,7,85.524286


### EXERCÍCIO (SLIDE 20) - Métricas por mês

In [24]:
banco_dados.sql("""
SELECT
    t.ano_mes,
    SUM(f.valor_total) AS faturamento_total,
    COUNT(f.order_id) AS qnt_pedidos,
    AVG(f.valor_total) AS ticket_medio
FROM fato_vendas f
INNER JOIN dim_tempo t
    ON f.tempo_sk = t.tempo_sk
GROUP BY t.ano_mes
ORDER BY 
    t.ano_mes ASC
""").df()

,ano_mes,faturamento_total,qnt_pedidos,ticket_medio
0,2016-09,143.46,3,47.820000
1,2016-10,46490.66,313,148.532460
2,2016-12,19.62,1,19.620000
3,2017-01,127482.37,913,139.630197
4,2017-02,271239.32,1858,145.984564
5,2017-03,414330.95,2897,143.020694
6,2017-04,390812.40,2569,152.126275
7,2017-05,566851.40,4004,141.571279
8,2017-06,490050.37,3489,140.455824
9,2017-07,566299.08,4416,128.238016


## Exercício 16  — Faturamento por estado

Retorne o valor_total por estado
Dica: use as funções SUM, COUNT, AVG, GROUP_BY e ORDER_BY e JOIN

In [25]:
## Resposta exercício 16

banco_dados.sql("""
SELECT
    c.customer_state AS cliente_estado,
    SUM(f.valor_total) AS faturamento_total,
    COUNT(f.order_id) AS qnt_pedidos,
    AVG(f.valor_total) AS ticket_medio
FROM fato_vendas f
INNER JOIN dim_cliente c
    ON f.cliente_sk = c.cliente_sk
GROUP BY 
    c.customer_state
ORDER BY
    c.customer_state
""").df()

,cliente_estado,faturamento_total,qnt_pedidos,ticket_medio
0,AC,19575.33,91,215.113516
1,AL,94172.49,427,220.544473
2,AM,27585.47,163,169.236012
3,AP,16141.81,81,199.281605
4,BA,591137.81,3683,160.504428
5,CE,266436.77,1426,186.842055
6,DF,346123.35,2355,146.973822
7,ES,317657.93,2225,142.767609
8,GO,334212.35,2277,146.777492
9,MA,147803.55,800,184.754438


## Exercício 17 — Top 10 categorias

Retorne uma tabela que mostre as top 10 categorias de maior faturamento total

In [26]:
## Resposta exercício 17

banco_dados.sql("""
-- escreva aqui seu código
""").df()

AttributeError: 'NoneType' object has no attribute 'df'

## Exercício 18 (Aula 2 semana 9) — Os 5 estados com maior ticket médio

Retorne uma tabela que mostre o ticket médio por estado
Dica: Use a função COUNT, SUM, AVG e HAVING

In [ ]:
## Resposta exercício 18

banco_dados.sql("""
-- SUA CONSULTA SQL AQUI
""").df()

## Exercício 19 (Aula 2 semana 9) — Evolução mensal de faturamento

Use função SUM, GROUP_BY, ORDER_BY E JOIN

In [ ]:
### Resposta exercício 19

banco_dados.sql("""
-- SUA CONSULTA SQL AQUI
""").df()

AttributeError: 'NoneType' object has no attribute 'df'

### EXERCÍCIOS SLIDE 27: PRATIQUE FILTROS DE GRUPO

#### Pedidos por estado

Enunciado: Liste os estados com mais de 500 pedidos únicos. Ordene do maior para o menor volume

#### Receita por categoria

Enunciado: Liste as categorias com receita total acima de R$ 50.000. Ordene pela receita.

#### Ticket médio por categoria

Enunciado: Use categoria como grupo e mostre apenas categorias com ticket médio acima de R$ 100.

# AULA 3 - SEMANA 9
- DATE_TRUNC
- EXTRACT

### SLIDE 32: Transformando datas em período

In [ ]:
banco_dados.sql("""
-- Conversão para data no SQL
    SELECT CAST(t.data AS DATE) AS data_pedido
        FROM dim_tempo t
        LIMIT 5
""").df()

,data_pedido
0,2017-10-02
1,2018-07-24
2,2018-08-08
3,2017-11-18
4,2018-02-13


In [ ]:
banco_dados.sql("""
-- Agrupamento no início do mês
    SELECT 
        DATE_TRUNC('month', CAST(t.data AS DATE)) AS mes_referencia
        FROM dim_tempo t
        LIMIT 5
""").df()

,mes_referencia
0,2017-10-01
1,2018-07-01
2,2018-08-01
3,2017-11-01
4,2018-02-01


### Slide 33 - EXTRACT: Conceito e Sintaxe
Extraindo partes da data

In [ ]:
banco_dados.sql("""
SELECT
    CAST(t.data AS DATE) AS data_pedido,
    EXTRACT(YEAR FROM CAST(t.data AS DATE)) AS ano,
    EXTRACT(MONTH FROM CAST(t.data AS DATE)) AS mes,
    EXTRACT(QUARTER FROM CAST(t.data AS DATE)) AS trimestre,
    FROM dim_tempo t
    LIMIT 10
""").df()

,data_pedido,ano,mes,trimestre
0,2017-10-02,2017,10,4
1,2018-07-24,2018,7,3
2,2018-08-08,2018,8,3
3,2017-11-18,2017,11,4
4,2018-02-13,2018,2,1
5,2017-07-09,2017,7,3
6,2017-05-16,2017,5,2
7,2017-01-23,2017,1,1
8,2017-07-29,2017,7,3
9,2017-07-13,2017,7,3


### SLIDE 34 EXTRACT: Totais por Ano e Análise de Sazonalidade
Receita por ano e sazonalidade por mês

In [ ]:
banco_dados.sql("""
    SELECT
        EXTRACT(YEAR FROM CAST(t.data AS DATE)) AS ano,
        COUNT(f.order_id) AS qtd_pedidos,
        SUM(f.valor_total) AS receita_anual
    FROM fato_vendas f
    INNER JOIN dim_tempo t
        ON f.tempo_sk = t.tempo_sk
    GROUP BY EXTRACT(YEAR FROM CAST(t.data AS DATE))
    ORDER BY ano ASC;
""").df()

,ano,qtd_pedidos,receita_anual
0,2016,317,46653.74
1,2017,49556,6921535.24
2,2018,60324,8451584.77


### Sazonalidade por mês

In [ ]:
banco_dados.sql("""
-- Sazonalidade por mês   
    SELECT
        EXTRACT(MONTH FROM CAST(t.data AS DATE)) AS mes,
        COUNT(f.order_id) AS qtd_pedidos,
        SUM(f.valor_total) AS receita_mensal
    FROM fato_vendas f
    INNER JOIN dim_tempo t
        ON f.tempo_sk = t.tempo_sk
    GROUP BY 1 -- agrupa pelo número da coluna mes
    ORDER BY mes ASC;
""").df()

,mes,qtd_pedidos,receita_mensal
0,1,8950,1205369.83
1,2,9376,1237407.73
2,3,10914,1534929.19
3,4,10396,1523691.33
4,5,11814,1695625.92
5,6,10499,1502028.66
6,7,11379,1594106.36
7,8,11939,1631324.00
8,9,4740,701220.95
9,10,5527,797607.67


In [ ]:
banco_dados.sql("""
    
    SELECT
        EXTRACT(MONTH FROM CAST(t.data AS DATE)) AS mes,
        COUNT(f.order_id) AS qtd_pedidos,
        SUM(f.valor_total) AS receita_mensal
    FROM fato_vendas f
    INNER JOIN dim_tempo t
        ON f.tempo_sk = t.tempo_sk
    GROUP BY 1 -- agrupa pelo número da coluna mes
    ORDER BY mes ASC;
""").df()

,mes,qtd_pedidos,receita_mensal
0,1,8950,1205369.83
1,2,9376,1237407.73
2,3,10914,1534929.19
3,4,10396,1523691.33
4,5,11814,1695625.92
5,6,10499,1502028.66
6,7,11379,1594106.36
7,8,11939,1631324.00
8,9,4740,701220.95
9,10,5527,797607.67
